In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.svm import SVC

# Load Titanic
df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

features = ["Pclass", "Sex", "Age", "Fare", "FamilySize", "IsAlone"]
X = df[features]
y = df["Survived"]

# Create Pipelines for 3 models
pipelines = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),
    "SVM": Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf'))
    ]),
    "Random Forest": Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
}

# Compare all pipelines
for name, pipeline in pipelines.items():
    scores = cross_val_score(pipeline, X, y, cv=5)
    print(f"{name}:")
    print(f"  Mean: {scores.mean():.3f}")
    print(f"  Std:  {scores.std():.3f}")
    print()

Logistic Regression:
  Mean: 0.794
  Std:  0.018

SVM:
  Mean: 0.823
  Std:  0.019

Random Forest:
  Mean: 0.819
  Std:  0.029



In [4]:
# Tune SVM pipeline with GridSearchCV
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf'))
])

# Parameters (note: use 'model__' prefix!)
param_grid = {
    'model__C': [0.1, 1, 10],
    'model__gamma': ['scale', 'auto']
}

grid = GridSearchCV(svm_pipeline, param_grid, cv=5)
grid.fit(X, y)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_.round(3))

Best Parameters: {'model__C': 1, 'model__gamma': 'scale'}
Best Score: 0.823
